# 🚀 Z-Image-Turbo Ultra-Leve (Edição SDNQ 4-bit)
Este notebook usa compressão matemática para rodar o modelo de 32GB em GPUs menores.

### 🔑 Configuração Obrigatória:
1. Vá no ícone da **Chave (Secrets)** à esquerda.
2. Adicione `HF_TOKEN` (Hugging Face) e `NGROK_TOKEN` (ngrok).
3. Ative o botão **"Notebook access"** para ambos.

In [ ]:
# 1. Instalação de pacotes específicos para 4-bit
!pip install -U git+https://github.com/huggingface/diffusers git+https://github.com/Disty0/sdnq
!pip install fastapi uvicorn pillow pyngrok huggingface_hub python-multipart

In [ ]:
# 2. Autenticação e Configuração
import os
import torch
import diffusers
from google.colab import userdata
from huggingface_hub import login
from pyngrok import ngrok
from sdnq.loader import apply_sdnq_options_to_model

# Login Hugging Face
hf_token = userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token

# Login ngrok
ngrok_token = userdata.get('NGROK_TOKEN')
if ngrok_token:
    ngrok.set_auth_token(ngrok_token)

print("✅ Autenticação concluída!")

In [ ]:
# 3. Carregamento do Modelo e Servidor API
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from PIL import Image
import base64
from io import BytesIO
import time
import uvicorn
import threading

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

print("⏳ Carregando Modelo SDNQ (Aprox. 8-10GB VRAM)...")

# Carrega o modelo comprimido
model_id = "Disty0/Z-Image-Turbo-SDNQ-uint4-svd-r32"
pipe = diffusers.ZImagePipeline.from_pretrained(model_id, torch_dtype=torch.float32, device_map="cuda")

# Aplica a quantização nos componentes pesados
pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

# Otimização de memória para evitar Crash
pipe.enable_vae_tiling()
pipe.enable_model_cpu_offload()

class GenerateRequest(BaseModel):
    prompt: str
    steps: int = 9
    aspect_ratio: str = "1:1"

ASPECT_RATIOS = {"1:1": (1024, 1024), "16:9": (1216, 704), "9:16": (704, 1216)}

@app.post("/generate")
async def generate(req: GenerateRequest):
    start_time = time.time()
    w, h = ASPECT_RATIOS.get(req.aspect_ratio, (1024, 1024))
    
    try:
        image = pipe(
            prompt=req.prompt,
            height=h, width=w,
            num_inference_steps=req.steps,
            guidance_scale=0.0
        ).images[0]
        
        buffered = BytesIO()
        image.save(buffered, format="JPEG", quality=85)
        img_str = base64.b64encode(buffered.getvalue()).decode()
        
        return {"image": f"data:image/jpeg;base64,{img_str}", "latency": f"{time.time() - start_time:.2f}s"}
    except Exception as e:
        print(f"🚨 Erro: {e}")
        raise HTTPException(status_code=500, detail=str(e))

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_api, daemon=True).start()
time.sleep(10)
public_url = ngrok.connect(8000).public_url
print(f"\n🚀 BACKEND ONLINE EM: {public_url}")